## Parte A

Extracción con pd/SQL que devuelve reviews que cumplan todas las condiciones

1. Son reviews verificadas. 
2. Se publicaron entre el 1 de mayo y el 31 de octubre de 2023, ambas fechas incluidas. 
3. El pasajero viajó en  Economy Class  o  Premium Economy . 
4. El pasajero  no  recomienda la aerolínea. 
5. El overall_rating es 2 o menos. 
6. El puntaje de la categoría  Value For Money  es 2 o menos. 

In [1]:
# Libraries
from contextlib import closing
from pathlib import Path
import sqlite3

import pandas as pd
from IPython.display import display


# Establece rutas para ejecutar desde raíz
db_path = (Path("data") / "airline_reviews.db").resolve()
sql_path = Path("sql") / "extract_reviews.sql"

if not db_path.is_file():
    raise FileNotFoundError(f"Database not found: {db_path}")

query = sql_path.read_text(encoding="utf-8")

# Conexión a la base de datos en modo solo lectura
db_uri = f"{db_path.as_uri()}?mode=ro"

with closing(sqlite3.connect(db_uri, uri=True)) as conn:
    reviews_df = pd.read_sql_query(query, conn)

# Validación (Sin interactuar directamente con el query).
assert reviews_df["review_id"].notna().all(), "Missing review IDs."
assert reviews_df["review_id"].is_unique, "Duplicate review IDs."

assert reviews_df["published_date"].between(
    "2023-05-01", "2023-10-31"
).all(), "Reviews outside the required date interval."

assert reviews_df["seat_type"].isin(
    ["Economy Class", "Premium Economy"]
).all(), "Unexpected seat type."

assert reviews_df["overall_rating"].notna().all()
assert reviews_df["overall_rating"].le(2).all()

assert reviews_df["value_for_money"].notna().all()
assert reviews_df["value_for_money"].le(2).all()

assert reviews_df["published_date"].is_monotonic_decreasing

# Muestra el número de reseñas seleccionadas y las primeras filas del DataFrame
print(f"Selected reviews: {len(reviews_df)}")
display(reviews_df.head())

if reviews_df.empty:
    print("No matching reviews. Check the catalog, date format, and source data.")

Selected reviews: 50


,review_id,published_date,reviewer_name,seat_type,traveller_type,origin_destination,aircraft_family,overall_rating,value_for_money,cabin_staff_service,header_body
0,12,2023-10-31,J Furlong,Economy Class,Couple Leisure,"Anchorage, Heathrow",NaN,1,1,3.0,“BA clearly does not care”\n\n1. Ground crew i...
1,23,2023-10-17,S Carter,Economy Class,Couple Leisure,NaN,NaN,1,1,3.0,“I hate British Airways!”\n\nI hate British Ai...
2,22,2023-10-17,A Gernal,Economy Class,Solo Leisure,"Geneva, London Heathrow",A320,1,2,2.0,gave up being a full-service airline\n\nThe fl...
3,30,2023-10-11,Glenn Taylor-Biffen,Economy Class,Couple Leisure,"Gatwick, Venice",NaN,2,1,NaN,BA Gatwick are not BA\n\nCaught up in the Gatw...
4,29,2023-10-11,Glenn Taylor-Biffen,Economy Class,Couple Leisure,"Catania, Gatwick",A320,1,1,1.0,Never again BA\n\nThe already delayed aircraft...
